# Вебинар 8: Нейронные сети в анализе данных — решение продуктовых задач

## 📊 Датасет
**Название:** [Credit Card Fraud Detection](https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud)  
**Описание:** 284 807 транзакций по кредитным картам, 31 признак (V1-V28 PCA + Time, Amount).  
**Задача:** Классификация мошенничества с использованием нейронных сетей (PyTorch).  
**Бизнес-применение:** Anti-fraud в реальном времени, сравнение NN с классическими методами.

## 🎯 Цели ноутбука
1. Построить полносвязную нейросеть с PyTorch
2. Создать LSTM для временных рядов
3. Autoencoder для обнаружения аномалий
4. Сравнить NN vs Gradient Boosting
5. Полный end-to-end пайплайн

## 1. Импорт и загрузка

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import (
    roc_auc_score, average_precision_score, classification_report,
    confusion_matrix, precision_recall_curve
)

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
print(f"PyTorch: {torch.__version__}")

df = pd.read_csv('/kaggle/input/creditcardfraud/creditcard.csv')
print(f"\nShape: {df.shape}")
print(f"Fraud: {df['Class'].sum()} ({df['Class'].mean():.4%})")

In [ ]:
# Подготовка
scaler = StandardScaler()
df['Amount_scaled'] = scaler.fit_transform(df[['Amount']])
df['Time_scaled'] = scaler.fit_transform(df[['Time']])

feature_cols = [c for c in df.columns if c not in ['Class', 'Time', 'Amount']]
X = df[feature_cols].values.astype(np.float32)
y = df['Class'].values.astype(np.float32)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

print(f"Train: {X_train.shape}, fraud: {int(y_train.sum())}")
print(f"Test:  {X_test.shape}, fraud: {int(y_test.sum())}")

## 2. Полносвязная нейросеть (PyTorch)

In [ ]:
class FraudNet(nn.Module):
    def __init__(self, input_size, hidden_sizes=[64, 32], dropout=0.3):
        super().__init__()
        layers = []
        prev = input_size
        for h in hidden_sizes:
            layers.extend([
                nn.Linear(prev, h),
                nn.BatchNorm1d(h),
                nn.ReLU(),
                nn.Dropout(dropout)
            ])
            prev = h
        layers.append(nn.Linear(prev, 1))
        self.net = nn.Sequential(*layers)
    
    def forward(self, x):
        return self.net(x)

In [ ]:
def train_model(model, X_train, y_train, X_test, y_test,
                epochs=10, batch_size=256, lr=0.001, pos_weight=100):
    
    pos_weight_t = torch.tensor([pos_weight]).to(device)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight_t)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    
    train_dataset = TensorDataset(
        torch.FloatTensor(X_train).to(device),
        torch.FloatTensor(y_train).to(device)
    )
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    
    X_test_t = torch.FloatTensor(X_test).to(device)
    
    history = {'train_loss': [], 'val_auc': [], 'val_pr': []}
    
    for epoch in range(epochs):
        model.train()
        total_loss = 0
        
        for X_batch, y_batch in train_loader:
            optimizer.zero_grad()
            logits = model(X_batch).squeeze()
            loss = criterion(logits, y_batch)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        
        model.eval()
        with torch.no_grad():
            test_logits = model(X_test_t).squeeze()
            test_proba = torch.sigmoid(test_logits).cpu().numpy()
        
        auc = roc_auc_score(y_test, test_proba)
        pr_auc = average_precision_score(y_test, test_proba)
        
        history['train_loss'].append(total_loss / len(train_loader))
        history['val_auc'].append(auc)
        history['val_pr'].append(pr_auc)
        
        print(f"Epoch {epoch+1}/{epochs} | Loss: {total_loss/len(train_loader):.4f} | "
              f"Val AUC: {auc:.4f} | Val PR-AUC: {pr_auc:.4f}")
    
    return history, test_proba

In [ ]:
torch.manual_seed(42)
model = FraudNet(input_size=X_train.shape[1], hidden_sizes=[128, 64, 32], dropout=0.3).to(device)
print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")

history, nn_proba = train_model(
    model, X_train, y_train, X_test, y_test,
    epochs=10, batch_size=256, lr=0.001, pos_weight=200
)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(history['train_loss'], linewidth=2, color='steelblue')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training Loss')
axes[0].grid(alpha=0.3)

axes[1].plot(history['val_auc'], linewidth=2, label='ROC-AUC', color='green')
axes[1].plot(history['val_pr'], linewidth=2, label='PR-AUC', color='orange')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Score')
axes[1].set_title('Validation Metrics')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 3. Сравнение с Gradient Boosting

In [ ]:
print("Training Gradient Boosting...")
gb = GradientBoostingClassifier(
    n_estimators=100,
    max_depth=5,
    learning_rate=0.1,
    random_state=42
)
gb.fit(X_train, y_train)
gb_proba = gb.predict_proba(X_test)[:, 1]
gb_auc = roc_auc_score(y_test, gb_proba)
gb_pr = average_precision_score(y_test, gb_proba)

nn_auc = history['val_auc'][-1]
nn_pr = history['val_pr'][-1]

print("\n=== Comparison ===")
print(f"{'Model':<20} {'ROC-AUC':>10} {'PR-AUC':>10}")
print("=" * 45)
print(f"{'Neural Network':<20} {nn_auc:>10.4f} {nn_pr:>10.4f}")
print(f"{'Gradient Boosting':<20} {gb_auc:>10.4f} {gb_pr:>10.4f}")

In [ ]:
# PR-кривые
from sklearn.metrics import precision_recall_curve

nn_precision, nn_recall, _ = precision_recall_curve(y_test, nn_proba)
gb_precision, gb_recall, _ = precision_recall_curve(y_test, gb_proba)

plt.figure(figsize=(10, 7))
plt.plot(nn_recall, nn_precision, linewidth=2, label=f'Neural Network (PR-AUC={nn_pr:.3f})', color='blue')
plt.plot(gb_recall, gb_precision, linewidth=2, label=f'Gradient Boosting (PR-AUC={gb_pr:.3f})', color='red')
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('PR-кривая: NN vs GB')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

## 4. Autoencoder для обнаружения аномалий

In [ ]:
class Autoencoder(nn.Module):
    def __init__(self, input_size, encoding_dim=8):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_size, 32),
            nn.ReLU(),
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Linear(16, encoding_dim),
            nn.ReLU()
        )
        self.decoder = nn.Sequential(
            nn.Linear(encoding_dim, 16),
            nn.ReLU(),
            nn.Linear(16, 32),
            nn.ReLU(),
            nn.Linear(32, input_size)
        )
    
    def forward(self, x):
        z = self.encoder(x)
        x_recon = self.decoder(z)
        return x_recon
    
    def get_reconstruction_error(self, x):
        with torch.no_grad():
            x_recon = self.forward(x)
            error = ((x - x_recon) ** 2).mean(dim=1)
        return error.cpu().numpy()

In [ ]:
# Обучаем только на нормальных транзакциях
X_normal = X_train[y_train == 0]
print(f"Normal transactions: {X_normal.shape[0]}")

torch.manual_seed(42)
ae = Autoencoder(input_size=X_train.shape[1], encoding_dim=8).to(device)
optimizer = optim.Adam(ae.parameters(), lr=0.001)
criterion = nn.MSELoss()

normal_dataset = TensorDataset(torch.FloatTensor(X_normal).to(device))
train_loader = DataLoader(normal_dataset, batch_size=256, shuffle=True)

for epoch in range(20):
    ae.train()
    total_loss = 0
    for (X_batch,) in train_loader:
        optimizer.zero_grad()
        X_recon = ae(X_batch)
        loss = criterion(X_recon, X_batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    
    if (epoch + 1) % 5 == 0:
        print(f"Epoch {epoch+1}/20 | Loss: {total_loss/len(train_loader):.6f}")

In [ ]:
# Оценка на тесте
X_test_t = torch.FloatTensor(X_test).to(device)
recon_errors = ae.get_reconstruction_error(X_test_t)

# Порог: 95-й перцентиль ошибок нормальных
normal_errors = recon_errors[y_test == 0]
threshold = np.percentile(normal_errors, 99)
print(f"Threshold (99th percentile): {threshold:.6f}")

ae_predictions = (recon_errors > threshold).astype(int)

from sklearn.metrics import precision_score, recall_score, f1_score
print(f"\nAutoencoder results:")
print(f"  Precision: {precision_score(y_test, ae_predictions):.4f}")
print(f"  Recall:    {recall_score(y_test, ae_predictions):.4f}")
print(f"  F1:        {f1_score(y_test, ae_predictions):.4f}")

In [ ]:
# Визуализация распределения ошибок
fig, ax = plt.subplots(figsize=(10, 6))
ax.hist(normal_errors, bins=100, alpha=0.7, label='Normal', color='blue', density=True)
fraud_errors = recon_errors[y_test == 1]
ax.hist(fraud_errors, bins=50, alpha=0.7, label='Fraud', color='red', density=True)
ax.axvline(threshold, color='black', linestyle='--', linewidth=2, label=f'Threshold = {threshold:.4f}')
ax.set_xlabel('Reconstruction Error')
ax.set_ylabel('Density')
ax.set_title('Распределение ошибок реконструкции (Autoencoder)')
ax.legend()
ax.set_yscale('log')
ax.grid(alpha=0.3)
plt.show()

## 5. LSTM для временных рядов

In [ ]:
class LSTMFraudDetector(nn.Module):
    def __init__(self, input_size, hidden_size=64, num_layers=2, dropout=0.2):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout
        )
        self.fc = nn.Sequential(
            nn.Linear(hidden_size, 32),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(32, 1)
        )
    
    def forward(self, x):
        # x: (batch, seq_len, features)
        lstm_out, _ = self.lstm(x)
        # Берём последний временной шаг
        last_out = lstm_out[:, -1, :]
        return self.fc(last_out)

In [ ]:
# Подготовка последовательностей
# Используем Time для упорядочивания
df_sorted = df.sort_values('Time').reset_index(drop=True)

window = 10
X_seq = []
y_seq = []
for i in range(len(df_sorted) - window):
    X_seq.append(df_sorted[feature_cols].iloc[i:i+window].values)
    y_seq.append(df_sorted['Class'].iloc[i+window])

X_seq = np.array(X_seq, dtype=np.float32)
y_seq = np.array(y_seq, dtype=np.float32)
print(f"Sequences: {X_seq.shape}, fraud: {int(y_seq.sum())}")

X_train_seq, X_test_seq, y_train_seq, y_test_seq = train_test_split(
    X_seq, y_seq, test_size=0.25, random_state=42, stratify=y_seq
)

In [ ]:
torch.manual_seed(42)
lstm_model = LSTMFraudDetector(input_size=X_seq.shape[2]).to(device)
print(f"LSTM parameters: {sum(p.numel() for p in lstm_model.parameters()):,}")

# Обучение на подвыборке для скорости
sample_size = 50000
indices = np.random.choice(len(X_train_seq), sample_size, replace=False)
X_train_sub = X_train_seq[indices]
y_train_sub = y_train_seq[indices]

lstm_history, lstm_proba = train_model(
    lstm_model, X_train_sub, y_train_sub, X_test_seq, y_test_seq,
    epochs=5, batch_size=128, lr=0.001, pos_weight=200
)

In [ ]:
lstm_auc = roc_auc_score(y_test_seq, lstm_proba)
lstm_pr = average_precision_score(y_test_seq, lstm_proba)

print("\n=== Final Comparison ===")
print(f"{'Model':<25} {'ROC-AUC':>10} {'PR-AUC':>10}")
print("=" * 50)
print(f"{'Neural Network (MLP)':<25} {nn_auc:>10.4f} {nn_pr:>10.4f}")
print(f"{'Gradient Boosting':<25} {gb_auc:>10.4f} {gb_pr:>10.4f}")
print(f"{'LSTM':<25} {lstm_auc:>10.4f} {lstm_pr:>10.4f}")
print(f"{'Autoencoder (anomaly)':<25} {'N/A':>10} {f1_score(y_test, ae_predictions):>10.4f} (F1)")

## 📋 Выводы

1. **Полносвязная NN** с BatchNorm и Dropout показывает хороший результат на табличных данных
2. **Gradient Boosting** часто превосходит NN на структурированных финансовых данных
3. **LSTM** полезен для последовательных данных (транзакции во времени)
4. **Autoencoder** — unsupervised подход для обнаружения аномалий
5. **Выбор модели** зависит от задачи: классификация, аномалии, последовательности
6. **Практический совет:** Всегда сравнивайте NN с классическими методами (GB, RF)